# 1. Setup Paths
Define the directory structure for the project, including workspaces, scripts, and model storage.

In [1]:
WORKSPACE_PATH = 'Tensorflow/workspace'
SCRIPTS_PATH = 'Tensorflow/scripts'
APIMODEL_PATH = 'Tensorflow/models'
ANNOTATION_PATH = WORKSPACE_PATH+'/annotations'
IMAGE_PATH = WORKSPACE_PATH+'/images'
MODEL_PATH = WORKSPACE_PATH+'/models'
PRETRAINED_MODEL_PATH = WORKSPACE_PATH+'/pre-trained-models'
CONFIG_PATH = MODEL_PATH+'/my_ssd_mobnet/pipeline.config'
CHECKPOINT_PATH = MODEL_PATH+'/my_ssd_mobnet/'

This verification step is only needed if you are using a specific virtual environment (like Conda) to manage library dependencies.

In [2]:
import sys
print(sys.executable)
#checking if the expected kernel is being utilized

C:\Users\DELL\.conda\envs\robot\python.exe


# 2. Create Label Map

### Define Classes
We map our object names ('Mask' and 'NoMask') to unique IDs. This file tells the model what it is looking for.

In [3]:
labels = [{'name' : 'Mask', 'id': 1}, {'name': 'NoMask', 'id':2 }]

In [4]:
labels

[{'name': 'Mask', 'id': 1}, {'name': 'NoMask', 'id': 2}]

Since this project focuses on mask detection, we only need these two specific labels.

This block converts our list into the `.pbtxt` format required by the TensorFlow Object Detection API.

In [5]:
with open(ANNOTATION_PATH + '/label_map.pbtxt', 'w') as f:
    for label in labels:
        f.write('item{\n')
        f.write('\tname:\'{}\'\n'.format(label['name']))
        f.write('\tid:{}\n'.format(label['id']))
        f.write('}\n')

# 3. Create TF Records

We now convert our images and annotations into `.record` files. This format allows TensorFlow to process the data more efficiently during training.

In [6]:
#creating record file for the training data
import sys

# This uses the exact path to your 'robot' environment's Python
!{sys.executable} {SCRIPTS_PATH + '/generate_tfrecord.py'} -x {IMAGE_PATH + '/train'} -l {ANNOTATION_PATH + '/label_map.pbtxt'} -o {ANNOTATION_PATH + '/train.record'}

Successfully created the TFRecord file: Tensorflow/workspace/annotations/train.record


C:\Users\DELL\.conda\envs\robot\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [7]:
#creating record file for the testing data
import sys

# This uses the exact path to your 'robot' environment's Python
!{sys.executable} {SCRIPTS_PATH + '/generate_tfrecord.py'} -x{IMAGE_PATH + '/test'} -l {ANNOTATION_PATH + '/label_map.pbtxt'} -o {ANNOTATION_PATH + '/test.record'}

Successfully created the TFRecord file: Tensorflow/workspace/annotations/test.record


C:\Users\DELL\.conda\envs\robot\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


# 4. Configure the Training Pipeline

We will customize a pre-trained model by updating its configuration file with our specific paths, class counts, and batch sizes.

Set a unique name for your custom model.

In [8]:
CUSTOM_MODEL_NAME = 'my_ssd_mobnet'

Create the model directory and copy the base configuration file from the pre-trained model.

In [9]:
import os
import shutil

os.makedirs(f"Tensorflow/workspace/models/{CUSTOM_MODEL_NAME}", exist_ok=True)

shutil.copy(
    f"{PRETRAINED_MODEL_PATH}/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/pipeline.config",
    f"{MODEL_PATH}/{CUSTOM_MODEL_NAME}/pipeline.config"
)

'Tensorflow/workspace/models/my_ssd_mobnet/pipeline.config'

In [10]:
#Importing necessary libraries to read and modify pipeline configurations
import tensorflow as tf
from object_detection.utils import config_util
from object_detection.protos import pipeline_pb2
from google.protobuf import text_format

C:\Users\DELL\.conda\envs\robot\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [11]:
CONFIG_PATH = MODEL_PATH + '/' + CUSTOM_MODEL_NAME + '/pipeline.config'
config = config_util.get_configs_from_pipeline_file(CONFIG_PATH)

Load the configuration file so we can modify it programmatically.

In [12]:
config #print empyt configure

{'model': ssd {
   num_classes: 90
   image_resizer {
     fixed_shape_resizer {
       height: 320
       width: 320
     }
   }
   feature_extractor {
     type: "ssd_mobilenet_v2_fpn_keras"
     depth_multiplier: 1.0
     min_depth: 16
     conv_hyperparams {
       regularizer {
         l2_regularizer {
           weight: 3.9999998989515007e-05
         }
       }
       initializer {
         random_normal_initializer {
           mean: 0.0
           stddev: 0.009999999776482582
         }
       }
       activation: RELU_6
       batch_norm {
         decay: 0.996999979019165
         scale: true
         epsilon: 0.0010000000474974513
       }
     }
     use_depthwise: true
     override_base_feature_extractor_hyperparams: true
     fpn {
       min_level: 3
       max_level: 7
       additional_layer_depth: 128
     }
   }
   box_coder {
     faster_rcnn_box_coder {
       y_scale: 10.0
       x_scale: 10.0
       height_scale: 5.0
       width_scale: 5.0
     }
   }
   matc

Parse the configuration file into a manageable object.

In [13]:
pipeline_config = pipeline_pb2.TrainEvalPipelineConfig()
with tf.io.gfile.GFile(CONFIG_PATH, 'r') as f:
    proto_str = f.read()
    text_format.Merge(proto_str, pipeline_config)

Update settings: set `num_classes` to 2, adjust `batch_size`, and link the correct paths for checkpoints and data.

In [14]:
pipeline_config.model.ssd.num_classes = 2
pipeline_config.train_config.batch_size = 4
pipeline_config.train_config.fine_tune_checkpoint = PRETRAINED_MODEL_PATH+'/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8/checkpoint/ckpt-0'
pipeline_config.train_config.fine_tune_checkpoint_type = "detection"
pipeline_config.train_input_reader.label_map_path= ANNOTATION_PATH + '/label_map.pbtxt'
pipeline_config.train_input_reader.tf_record_input_reader.input_path[:] = [ANNOTATION_PATH + '/train.record']
pipeline_config.eval_input_reader[0].label_map_path = ANNOTATION_PATH + '/label_map.pbtxt'
pipeline_config.eval_input_reader[0].tf_record_input_reader.input_path[:] = [ANNOTATION_PATH + '/test.record']

Save the modified configuration back to the file.

In [15]:
config_text = text_format.MessageToString(pipeline_config)
with tf.io.gfile.GFile(CONFIG_PATH, "wb") as f:
    f.write(config_text)

# 5. Train the Model

Generate the training command. You can run this in your terminal to begin the training process.

In [16]:
print("""python {}/research/object_detection/model_main_tf2.py --model_dir={}/{} --pipeline_config_path={}/{}/pipeline.config --num_train_steps=3000""".format(APIMODEL_PATH, MODEL_PATH, CUSTOM_MODEL_NAME, MODEL_PATH, MODEL_PATH, CUSTOM_MODEL_NAME))

python Tensorflow/models/research/object_detection/model_main_tf2.py --model_dir=Tensorflow/workspace/models/my_ssd_mobnet --pipeline_config_path=Tensorflow/workspace/models/Tensorflow/workspace/models/pipeline.config --num_train_steps=3000


**Note:** Run the generated command inside your project folder, `RealTimeObjectDetection`, to start training.